# 00 · Entorno y datos

Primer notebook que hay que correr. Verifica tres cosas antes de gastar tiempo en los demás:

1. **Versiones.** El congelado (`resultados/resultados_congelados.json`) se generó con Python 3.9.6, numpy 1.26.4, pandas 2.2.3 y scipy 1.13.1. Con otras versiones los números pueden diferir en los últimos decimales.
2. **Datos.** Que estén los archivos de `datos/MANIFIESTO_datos.csv`, con el tamaño correcto (y, si se pide, el SHA-256). Los de 2014 viajan en el repo; los de 2022 (unos 800 MB de INEGI) hay que colocarlos a mano: ver `datos/Data_2022/LEEME.md`.
3. **Referencia.** Qué commit y qué entorno produjeron el congelado que se usa como referencia.

Sin `Data_2022` se pueden correr solo los notebooks de 2014: 01 y 05 (y el 06 con `SOLO_2014 = True`).

In [ ]:
# Preparación (no modificar): ubica la raíz del repo, la usa como directorio de trabajo
# y pone codigo/ en el path. Funciona igual abriendo el notebook desde notebooks/ o desde
# la raíz, y también ejecutándolo con codigo/run_nb.py.
import os, sys
_raiz = os.getcwd()
while not os.path.isdir(os.path.join(_raiz, "codigo")) and os.path.dirname(_raiz) != _raiz:
    _raiz = os.path.dirname(_raiz)
os.chdir(_raiz)
sys.path.insert(0, os.path.join(_raiz, "codigo"))
print("Raíz del repo:", _raiz)

In [ ]:
import sys, platform, json
import numpy as np, pandas as pd, scipy

PINNED = {"numpy": "1.26.4", "pandas": "2.2.3", "scipy": "1.13.1"}
ACTUAL = {"numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__}
print("Python", platform.python_version(), "|", platform.platform())
for k, v in PINNED.items():
    print(f"  {k:<7}{ACTUAL[k]:<10}", "✓" if ACTUAL[k] == v else f"≠ {v} (versión del congelado)")

In [ ]:
import os, csv, hashlib
VERIFICAR_SHA256 = False        # True: además de nombre y tamaño, revisa el SHA-256 (tarda ~1 min en 2022)

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

filas = list(csv.DictReader(open("datos/MANIFIESTO_datos.csv", encoding="utf-8")))
resumen = {}
for fila in filas:
    ruta = os.path.join("datos", fila["carpeta"], fila["archivo"])
    if not os.path.exists(ruta):
        estado = "falta"
    elif os.path.getsize(ruta) != int(fila["bytes"]):
        estado = "tamaño distinto"
    elif VERIFICAR_SHA256 and sha256(ruta) != fila["sha256"]:
        estado = "SHA-256 distinto"
    else:
        estado = "ok"
    fila["estado"] = estado
    resumen.setdefault(fila["carpeta"], {}).setdefault(estado, 0)
    resumen[fila["carpeta"]][estado] += 1

for carpeta, cuenta in resumen.items():
    print(f"{carpeta:<15}", cuenta)
problemas = [f for f in filas if f["estado"] != "ok"]
print()
if not problemas:
    print("Datos completos: se pueden correr todos los notebooks.")
else:
    print(f"{len(problemas)} archivos con problema; primeros 10:")
    for f in problemas[:10]:
        print(f"  {f['carpeta']}/{f['archivo']}: {f['estado']}")
    if any(f["carpeta"] == "Data_2022" for f in problemas):
        print("\nFaltan datos de 2022: ver datos/Data_2022/LEEME.md (o `bash scripts/enlazar_datos_2022.sh <carpeta>`).")

In [ ]:
ref = json.load(open("resultados/resultados_congelados.json", encoding="utf-8"))["_meta"]
print("Congelado de referencia:")
print("  generado:", ref["fecha"])
print("  código  :", ref["git"])
print("  entorno :", ref["entorno"])